# 02 · Preparação da base de conhecimento

Executa as mesmas funções usadas por `scripts/build_knowledge_base.py`, mostrando o efeito de cada regra.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 200)

In [ ]:
from voc_rag.ingestion import load_raw_tables, build_review_base, clean_reviews, build_documents

tables = load_raw_tables(ROOT / "data/raw")
base = build_review_base(tables)
print(base.shape)
base.head(3)

In [ ]:
clean_df, report = clean_reviews(base)
pd.Series(report.to_dict()).to_frame("quantidade")

## Exemplos de normalização (título + mensagem)

In [ ]:
clean_df.loc[clean_df["text_source"] == "titulo_e_mensagem",
             ["review_comment_title", "review_comment_message", "text"]].head(8)

## Textos descartados por não serem informativos

In [ ]:
from voc_rag.ingestion.cleaning import normalize_text, combine_title_message, is_informative
tmp = base.drop_duplicates("review_id").copy()
tmp["text"] = [combine_title_message(normalize_text(t), normalize_text(m))
               for t, m in zip(tmp["review_comment_title"], tmp["review_comment_message"])]
tmp[(tmp["text"] != "") & ~tmp["text"].map(is_informative)]["text"].value_counts().head(20)

## Metadados derivados

In [ ]:
docs = build_documents(clean_df)
print(docs["delivery_status"].value_counts(), "\n")
print(docs["sentiment"].value_counts(), "\n")
docs["main_category"].value_counts().head(15)

In [ ]:
pd.crosstab(docs["delivery_status"], docs["review_score"], normalize="index").round(2)

**Leitura:** pedidos entregues com atraso concentram notas 1–2. Isso justifica oferecer o
filtro de **situação da entrega** como metadado — ele isola um recorte que o texto, sozinho,
nem sempre deixa explícito.